# In-Class Assignment 4/28: Naive Bayes Irrigation Model



In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    f1_score,
    recall_score,
)

RANDOM_STATE = 42

## Load and Prepare the Kaggle Train Data

The assignment asks us to train and test using only the Kaggle train data, so the original `train.csv` is split into a training portion and a validation portion. Categorical columns are converted with one-hot encoding so Gaussian Naive Bayes can use them.

In [7]:
train = pd.read_csv("../Kaggle/train.csv")

TARGET_COL = "Irrigation_Need"
ID_COL = "id"

feature_cols = [col for col in train.columns if col not in [ID_COL, TARGET_COL]]
X = pd.get_dummies(train[feature_cols], drop_first=False)
y_raw = train[TARGET_COL]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", train.shape)
print("Model feature shape:", X.shape)
print("Classes:", list(class_names))
print("Class proportions:")
print(y_raw.value_counts(normalize=True).round(4))

Train shape: (630000, 21)
Model feature shape: (630000, 43)
Classes: ['High', 'Low', 'Medium']
Class proportions:
Irrigation_Need
Low       0.5872
Medium    0.3795
High      0.0333
Name: proportion, dtype: float64


## Fit a Gaussian Naive Bayes Model

The baseline prediction uses the default probability rule: choose whichever class has the largest predicted probability.

In [8]:
nb_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", GaussianNB()),
    ]
)

nb_model.fit(X_train, y_train)

valid_probs = nb_model.predict_proba(X_valid)
baseline_pred = valid_probs.argmax(axis=1)

baseline_report = classification_report(
    y_valid,
    baseline_pred,
    target_names=class_names,
    zero_division=0,
)

print("Baseline classification report")
print(baseline_report)

Baseline classification report
              precision    recall  f1-score   support

        High       0.52      0.78      0.63      4202
         Low       0.88      0.80      0.83     73983
      Medium       0.70      0.76      0.73     47815

    accuracy                           0.78    126000
   macro avg       0.70      0.78      0.73    126000
weighted avg       0.80      0.78      0.79    126000



## Threshold the Rare `High` Class

`High` is the rarest irrigation need class, so I used it as the focus class. The table below shows how changing the probability threshold affects `High` recall, `High` F1, balanced accuracy, and the share of validation rows predicted as `High`.

In [9]:
focus_class = "High"
focus_idx = list(class_names).index(focus_class)
thresholds = np.arange(0.05, 0.51, 0.05)

rows = []
for threshold in thresholds:
    threshold_pred = baseline_pred.copy()
    threshold_pred[valid_probs[:, focus_idx] >= threshold] = focus_idx

    rows.append(
        {
            "threshold": threshold,
            "high_recall": recall_score(
                y_valid,
                threshold_pred,
                labels=[focus_idx],
                average="macro",
                zero_division=0,
            ),
            "high_f1": f1_score(
                y_valid,
                threshold_pred,
                labels=[focus_idx],
                average="macro",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(y_valid, threshold_pred),
            "predicted_high_rate": (threshold_pred == focus_idx).mean(),
        }
    )

threshold_results = pd.DataFrame(rows)
threshold_results

,threshold,high_recall,high_f1,balanced_accuracy,predicted_high_rate
0,0.05,0.880771,0.402304,0.763300,0.112675
1,0.10,0.857687,0.446288,0.769799,0.094833
2,0.15,0.841504,0.475748,0.772589,0.084627
3,0.20,0.829843,0.501691,0.774900,0.076976
4,0.25,0.819848,0.523437,0.776246,0.071119
5,0.30,0.812232,0.546211,0.778030,0.065833
6,0.35,0.802713,0.565039,0.778398,0.061405
7,0.40,0.792718,0.583567,0.778370,0.057254
8,0.45,0.785578,0.605022,0.779246,0.053254
9,0.50,0.776059,0.625492,0.779168,0.049405


## Compare Baseline vs. Selected Threshold

I selected a `High` threshold of `0.20`. This means any validation row with predicted probability of at least 20% for `High` is assigned to `High`, even if another class has a larger probability.

In [10]:
selected_threshold = 0.20

threshold_pred = baseline_pred.copy()
threshold_pred[valid_probs[:, focus_idx] >= selected_threshold] = focus_idx

baseline_metrics = {
    "prediction_rule": "default highest probability",
    "high_recall": recall_score(
        y_valid,
        baseline_pred,
        labels=[focus_idx],
        average="macro",
        zero_division=0,
    ),
    "high_f1": f1_score(
        y_valid,
        baseline_pred,
        labels=[focus_idx],
        average="macro",
        zero_division=0,
    ),
    "balanced_accuracy": balanced_accuracy_score(y_valid, baseline_pred),
    "predicted_high_rate": (baseline_pred == focus_idx).mean(),
}

threshold_metrics = {
    "prediction_rule": f"High probability >= {selected_threshold:.2f}",
    "high_recall": recall_score(
        y_valid,
        threshold_pred,
        labels=[focus_idx],
        average="macro",
        zero_division=0,
    ),
    "high_f1": f1_score(
        y_valid,
        threshold_pred,
        labels=[focus_idx],
        average="macro",
        zero_division=0,
    ),
    "balanced_accuracy": balanced_accuracy_score(y_valid, threshold_pred),
    "predicted_high_rate": (threshold_pred == focus_idx).mean(),
}

comparison = pd.DataFrame([baseline_metrics, threshold_metrics])
comparison

,prediction_rule,high_recall,high_f1,balanced_accuracy,predicted_high_rate
0,default highest probability,0.776059,0.625492,0.779168,0.049405
1,High probability >= 0.20,0.829843,0.501691,0.774900,0.076976


In [11]:
print("Baseline classification report")
print(
    classification_report(
        y_valid,
        baseline_pred,
        target_names=class_names,
        zero_division=0,
    )
)

print("Threshold classification report")
print(
    classification_report(
        y_valid,
        threshold_pred,
        target_names=class_names,
        zero_division=0,
    )
)

Baseline classification report
              precision    recall  f1-score   support

        High       0.52      0.78      0.63      4202
         Low       0.88      0.80      0.83     73983
      Medium       0.70      0.76      0.73     47815

    accuracy                           0.78    126000
   macro avg       0.70      0.78      0.73    126000
weighted avg       0.80      0.78      0.79    126000

Threshold classification report
              precision    recall  f1-score   support

        High       0.36      0.83      0.50      4202
         Low       0.88      0.80      0.83     73983
      Medium       0.68      0.70      0.69     47815

    accuracy                           0.76    126000
   macro avg       0.64      0.77      0.68    126000
weighted avg       0.78      0.76      0.77    126000



## Discussion

I chose the `High` irrigation need class because it is the rarest class in the training data, at only about 3.3% of the rows. Missing this class could also be important because a `High` irrigation need means the field may require more attention.

The baseline Naive Bayes rule chooses the class with the highest predicted probability. With that default rule, `High` recall was about 0.78 and `High` F1 was about 0.63. I selected a lower threshold of `0.20` for the `High` class, meaning that any row with at least a 20% predicted probability of `High` was assigned to `High`.

The threshold rule increased `High` recall to about 0.83, so the model caught more of the true `High` irrigation cases. The tradeoff was that `High` precision and F1 decreased because more rows were predicted as `High`, creating more false positives. Overall accuracy and weighted average scores also dropped slightly. This is a useful tradeoff if catching more high-irrigation cases matters more than avoiding false alarms.

Compared with my stronger existing models, such as LightGBM or CatBoost, Gaussian Naive Bayes is much simpler and faster but less flexible. It assumes features are conditionally independent within each class, which is unlikely for this irrigation data. The boosted tree models can capture nonlinear patterns and feature interactions better, so I would expect them to perform better for a Kaggle submission. Naive Bayes is still useful here because it gives probabilities quickly and makes the threshold tradeoff easy to study.